Applying Tversky transformations to environmental audio

In [1]:
# imports

import torch
from torch.utils.data import DataLoader
import torchvision.transforms as tt
import torchvision.models as models
import math

import torch.nn as nn

import torch.nn.functional as F

from pyha_analyzer.preprocessors import MelSpectrogramPreprocessors
from tqdm.notebook import tqdm

import numpy as np
import matplotlib.pyplot as plt

In [2]:
config = {
    "learning_rate": 2e-3,
    "learning_rate_decay": 0,
    "device": 'mps',
    "seed": 1
}

In [3]:
torch.manual_seed(config["seed"])

In [4]:
from datasets import load_dataset

mnist_dataset = load_dataset("mnist")

train = mnist_dataset["train"]
test = mnist_dataset["test"]

def transform(batch):
  t = tt.Compose([
    tt.Grayscale(num_output_channels=3),
    tt.PILToTensor(),
    tt.ConvertImageDtype(torch.float)
    ])
  
  batch['image'] = [t(img) for img in batch['image']]
  batch['label'] = F.one_hot(torch.tensor(batch['label']), num_classes=10)
  
  return batch

train.set_transform(transform)
test.set_transform(transform)

Using the latest cached version of the dataset since mnist couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'mnist' at /Users/anu/.cache/huggingface/datasets/mnist/mnist/0.0.0/77f3279092a1c1579b2250db8eafed0ad422088c (last modified on Wed Sep  3 12:18:07 2025).


In [5]:
class Tversky(nn.Module):
    """
    Similar to a fully-connected layer, but computes Tversky similarity instead
    """
    def __init__(
        self,
        in_features: tuple,
        out_features: int,
        num_features=256,
        feature_bank=None,
        phi=torch.mul,
        substract=True,
        device=None,
        dtype=None
    ) -> None:
        factory_kwargs = {"device": device, "dtype": dtype}
        super().__init__()
        
        self.phi = phi
        self.substract = substract
        
        if feature_bank is None:
            self.feature_bank = nn.Parameter(
                torch.empty((num_features,) + in_features, **factory_kwargs)
            )
        else:
            self.feature_bank = feature_bank
            
        self.prototypes = nn.Parameter(
                torch.empty( (out_features,) + in_features, **factory_kwargs)
            )
            
        self.alpha = nn.Parameter(torch.empty(1))
        self.beta = nn.Parameter(torch.empty(1))
        self.theta = nn.Parameter(torch.empty(1))

        self.reset_parameters()
        
    def reset_parameters(self) -> None:
        nn.init.uniform_(self.feature_bank)
        nn.init.uniform_(self.prototypes)
        nn.init.uniform_(self.alpha)
        nn.init.uniform_(self.beta)
        nn.init.uniform_(self.theta)
    
    def forward(self, input: torch.Tensor):
        
        a_f = (input.flatten(1, -1).unsqueeze(1) * self.feature_bank.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(1)
        p_f = (self.prototypes.flatten(1, -1).unsqueeze(1) * self.feature_bank.flatten(1, -1).unsqueeze(0)).sum(-1).unsqueeze(0)
        

        val = self.phi(a_f, p_f)
        mask = (torch.minimum(a_f, p_f) >= 0).int()
        intersection = self.theta * (val * mask).sum(-1)
        
        
        if self.substract:
            alpha_difference = -self.alpha * (a_f * ((a_f > 0) | (p_f <= 0)).int()).sum(-1)
            beta_difference = -self.beta * (p_f * ((p_f > 0) | (a_f <= 0)).int()).sum(-1)
        else:
            alpha_difference = -self.alpha * ((a_f - p_f) * ((a_f > 0) | (p_f > 0) | (a_f > p_f)).int()).sum(-1)
            beta_difference = -self.beta * ((p_f - a_f) * ((p_f > 0) | (a_f > 0) | (p_f > a_f)).int()).sum(-1)
        
        return intersection + alpha_difference + beta_difference
    
t = Tversky((4,), 10, 5)

A = torch.rand((2, 4))

t(A).shape

torch.Size([2, 10])

In [6]:
ResNet = models.resnet50(weights=None)
ResNet.fc = nn.Sequential(
    Tversky((ResNet.fc.in_features,),10, 20, substract=True)
)

# ResNet = models.resnet50(weights=None)
# ResNet.fc = nn.Linear(ResNet.fc.in_features, 10)

model = nn.Sequential(
    nn.Upsample(size=(224, 224), mode='bilinear', align_corners=False),
    ResNet
).to(config['device'])

In [9]:
optimizer = torch.optim.AdamW(model.parameters(), weight_decay=0, lr=config["learning_rate"], betas=(0.8, 0.999))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch : (1-config["learning_rate_decay"])**epoch)
metric = torch.nn.CrossEntropyLoss()
train_loader = DataLoader(mnist_dataset['train'], batch_size=8, num_workers=0, shuffle=True)

In [11]:
step, loss = 0, []

correct, total = 0, 0

for epoch in range(5):
    for data in tqdm(train_loader, desc=str(epoch), leave=False):
        img = data['image']

        optimizer.zero_grad()
        x = img.to(config["device"])
        
        pred = model(x)
        
        label = data['label'].to(config["device"], torch.float)
        
        score = metric(pred, label)
        score.backward()
        optimizer.step()
        
        loss.append(score.item())
        
        
        if step % 10 == 0:
            with torch.no_grad():
                print(np.mean(loss[-10:-1]))
                
                print(pred.argmax(dim=-1))
                print(label.argmax(dim=-1))
            
        step += 1

0:   0%|          | 0/7500 [00:00<?, ?it/s]

nan
tensor([6, 6, 6, 6, 6, 6, 6, 6], device='mps:0')
3079.7063802083335
tensor([8, 8, 8, 8, 8, 8, 8, 8], device='mps:0')
2398.3079427083335
tensor([8, 8, 8, 8, 8, 8, 8, 8], device='mps:0')
2578.2092488606772
tensor([3, 3, 3, 3, 3, 3, 3, 3], device='mps:0')
1142.1468794080947
tensor([2, 2, 2, 2, 2, 2, 2, 2], device='mps:0')
926.5703230963813
tensor([7, 7, 7, 7, 7, 7, 7, 7], device='mps:0')
793.1344333224827
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='mps:0')
1218.3441975911458
tensor([8, 8, 8, 8, 8, 8, 8, 8], device='mps:0')
788.2711334228516
tensor([8, 8, 8, 8, 8, 8, 8, 8], device='mps:0')
368.8755459255642
tensor([2, 3, 2, 3, 9, 3, 3, 2], device='mps:0')
1212.3836491902669
tensor([3, 7, 7, 7, 7, 3, 3, 3], device='mps:0')
1041.4959852430557
tensor([0, 0, 0, 0, 0, 0, 0, 0], device='mps:0')
1130.8221367730034
tensor([3, 8, 3, 7, 7, 8, 7, 7], device='mps:0')
521.7661624484592
tensor([4, 4, 1, 4, 1, 4, 1, 1], device='mps:0')
486.44000265333386
tensor([4, 4, 4, 0, 0, 4, 4, 0], device='mps:0')


KeyboardInterrupt: 